# 202. Chat Template：训练、推理、工具消息与缓存一致性怎样保证？

> **面试问题：怎样将角色消息编译成稳定 token 协议，区分训练/推理 generation prompt，并防止模板版本变化污染模型输入和 prefix cache？**

## 先给结论

面试中不能只背术语；需要把数学坐标、消息状态、协议顺序或模板字节流变成可检验的状态机。下列代码只使用标准库和小数组，明确教学 oracle 与生产替换点；它们不等同于真实模型效果、网络可靠性或正式安全认证。

## 一手资料

- [Hugging Face Chat Templates](https://huggingface.co/docs/transformers/chat_templating)
- [Chat Templates Guide](https://github.com/huggingface/blog/blob/main/chat-templates.md)
- [Toolformer](https://arxiv.org/abs/2302.04761)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "explicit-assertions", "production": "versioned-and-observed"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "explicit-assertions"  # 执行本行的状态、计算或校验逻辑。
assert "observed" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：chat template 是模型输入协议，不是展示格式

同一段对话换了 role token、终止符或 generation prompt，模型看到的 token 序列就变了。训练、离线评测、在线推理、缓存 key 和工具消息必须使用同一版本模板；否则很容易出现静默质量下降。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class ChatMessage:  # 执行本行的状态、计算或校验逻辑。
    role: str  # 执行本行的状态、计算或校验逻辑。
    content: str  # 执行本行的状态、计算或校验逻辑。
    call_id: str = ""  # 执行本行的状态、计算或校验逻辑。
messages = [ChatMessage("system", "你是订单助手"), ChatMessage("user", "查询 o-1"), ChatMessage("assistant", "我来查询")]  # 执行本行的状态、计算或校验逻辑。
assert messages[0].role == "system"  # 执行本行的状态、计算或校验逻辑。
assert messages[-1].content == "我来查询"  # 执行本行的状态、计算或校验逻辑。
assert len(messages) == 3  # 执行本行的状态、计算或校验逻辑。


## 2. 渲染器：把每个 role 编译为明确的控制 token

渲染函数必须对未知 role fail closed，而不能把它当 user 文本拼入。这里使用教学 marker；真实 checkpoint 的 token 字符串必须以模型卡、tokenizer 配置和训练数据为准。


In [ ]:
role_markers = {"system": "<|system|>", "user": "<|user|>", "assistant": "<|assistant|>", "tool": "<|tool|>"}  # 执行本行的状态、计算或校验逻辑。
def render_message(message):  # 执行本行的状态、计算或校验逻辑。
    if message.role not in role_markers:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("未知 role 不能进入模板")  # 执行本行的状态、计算或校验逻辑。
    suffix = f" id={message.call_id}" if message.call_id else ""  # 执行本行的状态、计算或校验逻辑。
    return f"{role_markers[message.role]}{suffix}\n{message.content}<|end|>\n"  # 执行本行的状态、计算或校验逻辑。
assert render_message(messages[0]).startswith("<|system|>")  # 执行本行的状态、计算或校验逻辑。
assert "<|end|>" in render_message(messages[1])  # 执行本行的状态、计算或校验逻辑。
assert "id=" not in render_message(messages[2])  # 执行本行的状态、计算或校验逻辑。


## 3. 训练与推理：唯一合法差异是 generation prompt

训练样本包含目标 assistant 回复，通常不应在末尾额外添加新的 assistant 起始标记；在线生成则需要该标记提示模型继续回答。两条路径都必须复用同一个 message renderer。


In [ ]:
def render_chat(messages, add_generation_prompt):  # 执行本行的状态、计算或校验逻辑。
    rendered = "".join(render_message(message) for message in messages)  # 执行本行的状态、计算或校验逻辑。
    return rendered + ("<|assistant|>\n" if add_generation_prompt else "")  # 执行本行的状态、计算或校验逻辑。
train_text = render_chat(messages, False)  # 执行本行的状态、计算或校验逻辑。
serve_text = render_chat(messages[:2], True)  # 执行本行的状态、计算或校验逻辑。
assert train_text.endswith("<|end|>\n")  # 执行本行的状态、计算或校验逻辑。
assert serve_text.endswith("<|assistant|>\n")  # 执行本行的状态、计算或校验逻辑。
assert "<|user|>" in train_text and "<|user|>" in serve_text  # 执行本行的状态、计算或校验逻辑。


## 4. 工具消息：tool result 必须绑定到已有 call id

工具调用和工具结果是对话协议的一部分，不能只把 JSON 当普通 user 文本。这里把 call id 放在 marker 属性中；生产可以用模型约定的特殊 token/JSON 结构，但也必须保留一一对应关系。


In [ ]:
tool_call = ChatMessage("assistant", "{\"name\":\"weather\"}", "call-7")  # 执行本行的状态、计算或校验逻辑。
tool_result = ChatMessage("tool", "{\"temperature_c\":28}", "call-7")  # 执行本行的状态、计算或校验逻辑。
tool_text = render_chat([tool_call, tool_result], False)  # 执行本行的状态、计算或校验逻辑。
assert "<|tool|> id=call-7" in tool_text  # 执行本行的状态、计算或校验逻辑。
assert tool_call.call_id == tool_result.call_id  # 执行本行的状态、计算或校验逻辑。
assert tool_text.count("call-7") == 2  # 执行本行的状态、计算或校验逻辑。


## 5. 模板版本：缓存与模型输入必须以渲染结果/版本为键

不能只按原始 messages 做 prefix cache，因为换模板会改变 token 序列。版本化模板、tokenizer 和 special token 集，并将其加入 cache/trace key，才能在热更新和回滚时避免错误命中。


In [ ]:
def template_key(template_version, tokenizer_version, rendered):  # 执行本行的状态、计算或校验逻辑。
    import hashlib  # 执行本行的状态、计算或校验逻辑。
    return hashlib.sha256(f"{template_version}|{tokenizer_version}|{rendered}".encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
key_v1 = template_key("chat-v1", "tok-v1", serve_text)  # 执行本行的状态、计算或校验逻辑。
assert len(key_v1) == 64  # 执行本行的状态、计算或校验逻辑。
assert key_v1 != template_key("chat-v2", "tok-v1", serve_text)  # 执行本行的状态、计算或校验逻辑。
assert key_v1 != template_key("chat-v1", "tok-v2", serve_text)  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：role、模板和 generation prompt 失配必须可检测

线上最危险的是模板不匹配却仍返回看似正常的答案。发布前应比较 golden rendering、special token id 和模板 fingerprint；检查失败时拒绝启用新 checkpoint 或走受控回退。


In [ ]:
def compatible(expected_template, actual_template, expected_prompt, actual_prompt):  # 执行本行的状态、计算或校验逻辑。
    return expected_template == actual_template and expected_prompt == actual_prompt  # 执行本行的状态、计算或校验逻辑。
assert compatible("chat-v1", "chat-v1", True, True)  # 执行本行的状态、计算或校验逻辑。
assert not compatible("chat-v1", "chat-v2", True, True)  # 执行本行的状态、计算或校验逻辑。
assert not compatible("chat-v1", "chat-v1", True, False)  # 执行本行的状态、计算或校验逻辑。


## 7. Golden tests：以字节级模板输出做回归

模型输出会有随机性，但模板渲染应是确定的。为 system/user/assistant/tool、空内容、多轮和 generation prompt 保存 golden cases；任何修改先 diff 输出，再决定是否需要数据迁移或重新评测。


In [ ]:
golden = {"single_user": "<|user|>\nhi<|end|>\n<|assistant|>\n"}  # 执行本行的状态、计算或校验逻辑。
rendered = render_chat([ChatMessage("user", "hi")], True)  # 执行本行的状态、计算或校验逻辑。
assert rendered == golden["single_user"]  # 执行本行的状态、计算或校验逻辑。
assert len(golden) == 1  # 执行本行的状态、计算或校验逻辑。
assert rendered.count("<|assistant|>") == 1  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：训练、服务与缓存应共享模板指纹

一次请求的可复放制品至少需要 checkpoint、tokenizer、template、generation-prompt 规则、工具 schema 与渲染后输入摘要。这样可以定位质量变化来自模型权重还是消息格式。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"checkpoint": "demo-1", "tokenizer": "tok-v1", "template": "chat-v1", "generation_prompt": True, "input_key": key_v1}  # 执行本行的状态、计算或校验逻辑。
artifact_hash = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["template"] == "chat-v1"  # 执行本行的状态、计算或校验逻辑。
assert artifact["generation_prompt"] is True  # 执行本行的状态、计算或校验逻辑。
assert len(artifact_hash) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时先说明不变量，再给出主路径和失败分支，最后说明指标、版本制品与生产替换点。不要把一个受控样例的通过误报成模型质量、可靠网络或端到端安全保证。
